# 영어 Word2Vec 만들기

In [ ]:
import re
import urllib.request
import zipfile
from lxml import etree
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

In [ ]:
# nltk.download()

In [ ]:
# urlretrieve() : 특정 url에 파일 다운로드함
urllib.request.urlretrieve("https://raw.githubusercontent.com/ukairia777/tensorflow-nlp-tutorial/main/09.%20Word%20Embedding/dataset/ted_en-20160408.xml", filename="ted_en-20160408.xml")

In [ ]:
targetXML = open('ted_en-20160408.xml', 'r', encoding='UTF8')
target_text = etree.parse(targetXML)
target_text

In [ ]:
parse_text = '\n'.join(target_text.xpath('//content/text()'))
parse_text

In [ ]:
content_text = re.sub(r'\([^)]*\)','',parse_text) # 괄호 안의 모든 내용을 ''로 변환
content_text

In [ ]:
sent_text = sent_tokenize(content_text)
sent_text

In [ ]:
normalized_text = []
for string in sent_text:
    tokens = re.sub(r"[^a-z0-9]+", " ", string.lower()) # 소문자와 숫자를 제외한 문자들 공백으로 처리
    normalized_text.append(tokens)

In [ ]:
normalized_text

In [ ]:
result = [word_tokenize(sentence) for sentence in normalized_text]
print('총 샘플의 개수 : {}'.format(len(result)))

In [ ]:
result[:100]

In [ ]:
for line in result[:3]:
    print(line)

In [ ]:
# pip install gensim
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

# sentences : 학습 데이터, vector_size : 하나의 단어를 몇 차원의 숫자(벡터)로 표현할 것인가 설정, window : 컨텍스트 윈도우 크기(앞뒤로 n개의 단어),
# min_count :데이터 전체에서 최소 5번 이상 등장한 단어만 학습에 포함시킴, workers : cpu 코어 사용 갯수, sg : 0 = CBOW, 1 = Skip-gram
model = Word2Vec(sentences=result, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [ ]:
model_result = model.wv.most_similar("man")
print(model_result)

In [ ]:
model.wv.save_word2vec_format('eng_w2v')

In [ ]:
loaded_model = KeyedVectors.load_word2vec_format("eng_w2v")

In [ ]:
model_result = loaded_model.most_similar('man')
print(model_result)

# 네이버 영화 리뷰 한국어 Word2Vec 만들기

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="ratings.txt")

In [ ]:
import pandas as pd

train_data = pd.read_table('ratings.txt')
train_data[:5]

In [ ]:
print(len(train_data))

In [ ]:
print(train_data.isnull().values.any())

In [ ]:
train_data = train_data.dropna(how='any')
print(train_data.isnull().values.any())

In [ ]:
print(len(train_data))

In [ ]:
train_data['document'] = train_data['document'].str.replace(r"[^ㄱ-ㅎㅏ-ㅣ가-힣 ]",r"", regex=True) # 한글과 공백 제외 삭제
train_data[:5]

In [ ]:
from konlpy.tag import Okt
from tqdm import tqdm

stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

okt = Okt()
tokenized_data = []
for sentence in tqdm(train_data['document']): # pip install tqdm
    tokenized_sentence = okt.morphs(sentence, stem=True) # stem = True : 동사나 형용사의 기본형을 찾음
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords]
    tokenized_data.append(stopwords_removed_sentence)

In [ ]:
import matplotlib.pyplot as plt

print("리뷰의 최대 길이 :",max(len(review) for review in tokenized_data))
print("리뷰의 평균 길이 :",sum(map(len, tokenized_data))/len(tokenized_data))
plt.hist([len(review) for review in tokenized_data], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [ ]:
model.wv.vectors.shape # 임베딩 벡터 크기

In [ ]:
print(model.wv.most_similar("최민식"))

In [ ]:
print(model.wv.most_similar("히어로"))

#  20뉴스그룹 데이터 : SGNS

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.datasets import fetch_20newsgroups
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
dataset = fetch_20newsgroups(shuffle=True, random_state=1, remove=('headers','footers','quotes'))
documents = dataset.data
print("총 샘플 수 :",len(documents))

In [ ]:
news_df = pd.DataFrame({'document':documents})
news_df['clean_doc'] = news_df['document'].str.replace("[^a-zA-Z]"," ")
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x : ' '.join([w for w in x.split() if len(w) > 3])) # 각 단어의 길이가 3보다 작으면 삭제됨
news_df['clean_doc'] = news_df['clean_doc'].apply(lambda x : x.lower())

In [ ]:
news_df.isnull().values.any()

In [ ]:
news_df.replace("", float("NaN"), inplace = True)
news_df.isnull().values.any()

In [ ]:
news_df.dropna(inplace = True)
print("총 샘플 수 :",len(news_df))

In [ ]:
stop_words = stopwords.words('english')
tokenized_doc = news_df['clean_doc'].apply(lambda x:x.split())
tokenized_doc = tokenized_doc.apply(lambda x: [item for item in x if item not in stop_words])
tokenized_doc = tokenized_doc.to_list()

In [ ]:
drop_train = [index for index, sentence in enumerate(tokenized_doc) if len(sentence) <= 1]
tokenized_doc = np.array(tokenized_doc, dtype=object)
tokenized_doc = np.delete(tokenized_doc, drop_train, axis=0)
print('총 샘플 수 :',len(tokenized_doc))

In [ ]:
tokenized_doc.shape

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(tokenized_doc)

word2idx = tokenizer.word_index
idx2word = {value : key for key, value in word2idx.items()}
encoded = tokenizer.texts_to_sequences(tokenized_doc)

In [ ]:
print(encoded[:2])

In [ ]:
vocab_size = len(word2idx) + 1
print("단어 집합의 크기 :", vocab_size)

### 네거티브 샘플링을 통한 데이터셋 구성

In [ ]:
# from tensorflow.keras.preprocessing import skipgrams
from tensorflow.keras.preprocessing.sequence import skipgrams

skip_grams = [skipgrams(sample, vocabulary_size=vocab_size, window_size=10) for sample in encoded[:10]]

In [ ]:
pairs, labels = skip_grams[0][0], skip_grams[0][1]

for i in range(5):
    print("({:s} ({:d}), {:s} ({:d}) -> {:d}".format(
        idx2word[pairs[i][0]], pairs[i][0],
        idx2word[pairs[i][1]], pairs[i][1],
        labels[i]))

In [ ]:
print('전체 샘플 수 :',len(skip_grams))

In [ ]:
print(len(pairs))
print(len(labels))

In [ ]:
skip_grams = [skipgrams(sample, vocabulary_size=vocab_size, window_size=10) for sample in encoded]

In [ ]:
len(skip_grams[0][1])

In [ ]:
skip_grams[0][1]

In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Reshape, Activation, Input
from tensorflow.keras.layers import Dot
from tensorflow.keras.utils import plot_model
from IPython.display import SVG

In [ ]:
embedding_dim = 100
w_inputs = Input(shape=(1, ), dtype='int32')
word_embedding = Embedding(vocab_size, embedding_dim)(w_inputs) # 중심단어에 대한 임베딩 벡터

c_inputs = Input(shape=(1,), dtype='int32')
context_embedding = Embedding(vocab_size, embedding_dim)(c_inputs) # 주변단어에 대한 임베딩 벡터

In [ ]:
import numpy as np
import tensorflow as tf

a = np.array([[[1,10], [100,1000]]])
b = np.array([[[1,2], [3,4]]])

print(tf.keras.layers.dot([a,b], axes=(2)))

In [ ]:
dot_product = Dot(axes=2)([word_embedding, context_embedding])
dot_product = Reshape((1,), input_shape=(1, 1))(dot_product)
output = Activation('sigmoid')(dot_product)

model = Model(inputs=[w_inputs, c_inputs], outputs=output)
model.summary()
model.compile(loss='binary_crossentropy', optimizer='adam')

In [ ]:
for epoch in range(1,6):
    loss = 0
    for _, elem in enumerate(skip_grams):
        first_elem = np.array(list(zip(*elem[0]))[0], dtype='int32') # 중심 단어
        second_elem = np.array(list(zip(*elem[0]))[1], dtype='int32') # 주변 단어
        labels = np.array(elem[1], dtype='int32') # 정답
        X = [first_elem, second_elem]
        Y = labels
        loss += model.train_on_batch(X,Y) # 학습
    print("epoch :",epoch, 'Loss :',loss)

In [ ]:
import gensim

f = open('vectors.txt', 'w', encoding='utf-8') # 임베딩 벡터를 저장할 파일 오픈
f.write('{} {}\n'.format(vocab_size-1, embedding_dim))
vectors = model.get_weights()[0] # 임베딩 벡터의 가중치 저장
for word, i in tokenizer.word_index.items():
    f.write("{} {}\n".format(word,  ' '.join(map(str,list(vectors[i,:]))))) # 임베딩 벡터를 저장
f.close()

w2v = gensim.models.KeyedVectors.load_word2vec_format('./vectors.txt', binary=False) # 저장된 임베딩 벡터 로드

In [ ]:
w2v.most_similar(positive=['soldiers'])

In [ ]:
w2v.most_similar(positive=['doctor'])

# 패스트 텍스트

In [ ]:
import urllib.request
import zipfile
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/GaoleMeng/RNN-and-FFNN-textClassification/master/ted_en-20160408.xml", filename="ted_en-20160408.xml")

In [ ]:
targetXML = open('ted_en-20160408.xml', 'r', encoding='UTF8')
target_text = etree.parse(targetXML)
parse_text = '\n'.join(target_text.xpath('//content/text()'))
content_text = re.sub(r'\([^)]*\)','',parse_text)
sent_text = sent_tokenize(content_text)

normalized_text = []
for string in sent_text:
    tokens = re.sub(r"[^a-z0-9]+", " ", string.lower())
    normalized_text.append(tokens)

result = [word_tokenize(sentence) for sentence in normalized_text]

In [ ]:
print("총 샘플의 개수 : {}".format(len(result)))

In [ ]:
from gensim.models import Word2Vec, FastText

In [ ]:
model = Word2Vec(sentences=result, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [ ]:
model.wv.most_similar("electrofishing") # Word2Vec에서는 없는 단어는 오류 발생

In [ ]:
model = FastText(result, vector_size=100, window=5, min_count=5, workers=4, sg=1)

model.wv.most_similar("electrofishing") # 없는 단어지만 유사한 단어를 찾음

# 자모 단위 한국어 패스트 텍스트 학습

In [ ]:
import re
import pandas as pd
import urllib.request
from tqdm import tqdm
import hgtk # pip install hgtk
from konlpy.tag import Mecab

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt", filename="ratings_total.txt")

In [ ]:
total_data = pd.read_table('ratings_total.txt', names=['ratings', 'reviews'])
print('전체 리뷰 개수 :',len(total_data))

In [ ]:
total_data[:5]

### hgtk 튜토리얼

In [ ]:
hgtk.checker.is_hangul('ㄱ')

In [ ]:
hgtk.checker.is_hangul('28')

In [ ]:
hgtk.letter.decompose('남')

In [ ]:
hgtk.letter.compose('ㄴ','ㅏ')

In [ ]:
hgtk.letter.compose('ㄴ','ㅏ','ㅁ')

In [ ]:
hgtk.letter.decompose('1') # 한글 아니면 오류

In [ ]:
hgtk.letter.compose('ㄴ','ㅁ','ㅁ') # 순서가 맞지 않거나 모음이 없으면 오류

In [ ]:
def word_to_jamo(token):
    def to_special_token(jamo):
        if not jamo:
            return '-'
        else:
            return jamo

    decomposed_token = ''
    for char in token:
        try:
            cho, jung, jong = hgtk.letter.decompose(char)
            cho = to_special_token(cho)
            jung = to_special_token(jung)
            jong = to_special_token(jong)
            decomposed_token = decomposed_token + cho + jung + jong

        except Exception as exception:
            if type(exception).__name__ == 'NotHangulException':
                decomposed_token += char

    return decomposed_token

In [ ]:
word_to_jamo("남동생")

In [ ]:
word_to_jamo("여동생")

In [ ]:
mecab = Mecab(r"C:\mecab\mecab-ko-dic")

In [ ]:
print(mecab.morphs("선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다."))

In [ ]:
def tokenize_by_jamo(s):
    return [word_to_jamo(token) for token in mecab.morphs(s)]

In [ ]:
print(tokenize_by_jamo('선물용으로 빨리 받아서 전달했어야 하는 상품이었는데 머그컵만 와서 당황했습니다.'))

In [ ]:
tokenized_data = []
for sample in tqdm(total_data['reviews'].to_list()):
    tokenized_sample = tokenize_by_jamo(sample)
    tokenized_data.append(tokenized_sample)

In [ ]:
len(tokenized_data)

In [ ]:
tokenized_data[0]

In [ ]:
def jamo_to_word(jamo_sequence):
    tokenized_jamo = []
    index = 0
    while index < len(jamo_sequence):
        if not hgtk.checker.is_hangul(jamo_sequence[index]):
            tokenized_jamo.append(jamo_sequence[index])
            index = index + 1
        else:
            tokenized_jamo.append(jamo_sequence[index:index + 3])
            index = index + 3

    word = ''
    try:
        for jamo in tokenized_jamo:
            if len(jamo) == 3:
                if jamo[2] == "-":
                    word = word + hgtk.letter.compose(jamo[0], jamo[1])
                else:
                    word = word + hgtk.letter.compose(jamo[0], jamo[1], jamo[2])
            else:
                word = word + jamo
    except Exception as exception:  
        if type(exception).__name__ == 'NotHangulException':
            return jamo_sequence

    return word

In [ ]:
jamo_to_word('ㄴㅏㅁㄷㅗㅇㅅㅐㅇ')

In [ ]:
jamo_to_word('ㅇㅕ-ㄷㅗㅇㅅㅐㅇ')

In [ ]:
import fasttext # pip install fasttext-wheel

In [ ]:
with open('tokenized_data.txt','w',encoding='utf-8') as out:
    for line in tqdm(tokenized_data, unit='line'):
        out.write(' '.join(line) + '\n')

In [ ]:
model = fasttext.train_unsupervised('tokenized_data.txt',model='cbow') # model='skipgram', 학습, dim=100

In [ ]:
model.save_model("fasttext.bin") # 모델 저장

In [ ]:
model = fasttext.load_model("fasttext.bin") # 모델 로드

In [ ]:
model[word_to_jamo('남동생')] # 남동생에 대한 임베딩 벡터 100차원 출력

In [ ]:
model.get_nearest_neighbors(word_to_jamo('남동생'), k=5)

In [ ]:
def transform(word_sequence):
    return [(jamo_to_word(word), similarity) for (similarity, word) in word_sequence]

In [ ]:
print(transform(model.get_nearest_neighbors(word_to_jamo('남동생'), k = 10)))

# 사전 훈련된 워드 임베딩

## 케라스 임베딩

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
sentences = ['nice great best amazing', 'stop lies', 'pitiful nerd', ' excellent work', 'supreme quality', 'bad', 'highly respectable']
y_train = [1, 0, 0, 1, 1, 0, 1]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1
print('단어 집합 :',vocab_size)

In [ ]:
X_encoded = tokenizer.texts_to_sequences(sentences)
print('정수 인코딩 결과 :',X_encoded)

In [ ]:
max_len = max(len(l) for l in X_encoded)
print('최대 길이 :',max_len)

In [ ]:
X_train = pad_sequences(X_encoded, maxlen=max_len, padding='post')

In [ ]:
y_train = np.array(y_train)
print("패딩 결과 :")
print(X_train)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten

embedding_dim = 4

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_len))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
model.fit(X_train,y_train,epochs=100, verbose=2)

## 사전 훈련된 Word2Vec 사용

In [ ]:
import gensim

word2vec_model = gensim.models.KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True)
print("모델의 크기(shape) :",word2vec_model.vectors.shape)

In [ ]:
embedding_matrix = np.zeros((vocab_size, 300)) # vocab_size : 이전 예제 자료
print('임베딩 행렬의 크기(shape) :',np.shape(embedding_matrix))

In [ ]:
def get_vector(word):
    if word in word2vec_model:
        return word2vec_model[word]
    else:
        return None

In [ ]:
tokenizer.word_index

In [ ]:
for word, index in tokenizer.word_index.items():
    vector_value = get_vector(word)
    if vector_value is not None:
        embedding_matrix[index] = vector_value

In [ ]:
print(word2vec_model['nice'])

In [ ]:
print(len(word2vec_model['nice']))

In [ ]:
print('단어 nice의 맵핑된 정수 :',tokenizer.word_index['nice'])

In [ ]:
print(embedding_matrix[1])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten, Input


model = Sequential()
model.add(Input(shape=(max_len,),dtype='int32'))
e = Embedding(vocab_size, 300, weights=[embedding_matrix], input_length=max_len, trainable=False)
model.add(e)
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
model.fit(X_train,y_train,epochs=100, verbose=2)